<a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_12/fl/server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning mit Flower AI — Server-Notebook 🌸

Dieses Notebook läuft **einmal** und wird von der Dozentin / dem Dozenten in Google Colab gestartet.
Es übernimmt die Rolle des **Flower-Servers**: Es koordiniert bis zu 20 Studierenden-Clients, die
jeweils ihr eigenes `client.ipynb` ausführen, und aggregiert deren lokal trainierte Modelle mittels
**Federated Averaging (FedAvg)** zu einem gemeinsamen globalen Modell.

## Architektur

```
                     Google Colab (Dozent:in)
                        Flower Server (FedAvg)
                                |
                          ngrok-Tunnel
                                |
      ------------------------------------------------------
      |        |        |        |       ...       |
     C0       C1       C2       C3               C19
                Google Colab Clients (Studierende)
```

## Ablauf für die Veranstaltung

1. Dieses Notebook von oben nach unten ausführen.
2. Die von `pyngrok` ausgegebene **öffentliche Server-Adresse** kopieren.
3. Diese Adresse den Studierenden mitteilen (z. B. per Chat/Whiteboard).
4. Die Studierenden tragen die Adresse und ihre individuelle `CLIENT_ID` (0–19) in ihr
   `client.ipynb` ein und führen es aus.
5. Sobald genügend Clients verbunden sind (siehe `min_available_clients`), startet das Training
   automatisch für die konfigurierte Anzahl an Kommunikationsrunden.

> **Hinweis zur Flower-API:** Dieses Projekt verwendet bewusst die adressbasierte
> `start_server()` / `start_client()`-API (bekannt als *Compat*- bzw. *Legacy*-API) anstelle des
> neuen SuperLink/SuperNode-Deployment-Runtimes (`flwr run`). Der Grund: Der neue Runtime ist auf
> dauerhaft betriebene Infrastruktur mit eigenständig gestarteten SuperNode-Prozessen ausgelegt und
> eignet sich nicht für 20 unabhängig voneinander gestartete Colab-Notebooks in einer einzelnen
> 90-minütigen Sitzung. Die hier verwendete API ist weiterhin Teil der öffentlichen Flower-API,
> erzeugt beim Import lediglich einen Deprecation-Hinweis in den Logs — dieser kann für dieses
> Lehrszenario ignoriert werden.

## 1. Installation

Installiert alle benötigten Pakete in der Colab-Laufzeit.

In [ ]:
!pip install -q "flwr>=1.32" tensorflow pyngrok matplotlib numpy

## 2. Gemeinsame Hilfsfunktionen (`common.py`)

Damit dieses Notebook **ohne externe Abhängigkeiten** direkt in Colab läuft, wird der Inhalt der
Projektdatei `common.py` (Modell, Datenpartitionierung, Visualisierung) hier per `%%writefile`
materialisiert. Wer das Projekt stattdessen per `git clone` lokal ausführt, kann diese Zelle
überspringen — die Datei `common.py` liegt dann bereits im Projektordner.

In [ ]:
!wget -O common.py https://raw.githubusercontent.com/dgaida/wpf_dlml_th_public/main/assets/exercises/week_12/fl/common.py

## 3. Zentrale Konfiguration

Alle Stellschrauben des Experiments sind in `common.FLConfig` gebündelt. Für den Server sind vor
allem folgende Felder relevant:

- **`num_rounds`**: Anzahl der Kommunikationsrunden (Standard: 10).
- **`fraction_fit`**: Anteil der verfügbaren Clients, die pro Runde **trainieren**
  (z. B. `0.5` = nur die Hälfte der verbundenen Clients trainiert pro Runde mit).
- **`fraction_evaluate`**: Anteil der verfügbaren Clients, die pro Runde an der föderierten
  **Evaluation** teilnehmen.
- **`min_fit_clients`**: Mindestanzahl an Clients, die für eine Trainingsrunde bereitstehen müssen.
- **`min_available_clients`**: Mindestanzahl an Clients, die insgesamt verbunden sein müssen,
  bevor überhaupt eine Runde gestartet wird — blockiert den Start, bis genug Studierende bereit
  sind.
- **`local_epochs`**: Wird als Trainingskonfiguration an jeden Client gesendet
  (siehe `on_fit_config_fn` weiter unten).

In [ ]:
import common
from common import FLConfig

# Zentrale Konfiguration -- hier für die Veranstaltung anpassen.
CONFIG = FLConfig(
    num_clients=20,
    num_rounds=10,
    local_epochs=1,
    fraction_fit=1.0,          # alle verbundenen Clients trainieren pro Runde mit
    fraction_evaluate=1.0,     # alle verbundenen Clients evaluieren pro Runde mit
    min_fit_clients=2,         # zum Testen niedrig; für die echte Veranstaltung z. B. 10
    min_available_clients=2,   # zum Testen niedrig; für die echte Veranstaltung z. B. 10
    partition_strategy="iid",  # "iid", "shard" oder "dominant" -- muss mit den Clients übereinstimmen!
)
print(CONFIG)

## 4. Modell und Startgewichte

Der Server benötigt eine Instanz des Modells, um daraus die **initialen globalen Gewichte** zu
erzeugen, mit denen alle Clients ihre erste lokale Trainingsrunde beginnen.

In [ ]:
import flwr as fl
from flwr.common import ndarrays_to_parameters

initial_model = common.create_model()
initial_parameters = ndarrays_to_parameters(initial_model.get_weights())
print("Modell erzeugt, Parameteranzahl:", len(initial_model.get_weights()))

## 5. ngrok-Tunnel starten

Colab-Instanzen sind von außen nicht erreichbar. `pyngrok` öffnet einen öffentlichen TCP-Tunnel
zum lokalen Port `8080`, über den die Studierenden-Clients den Server erreichen.

**Voraussetzung:** Ein kostenloser ngrok-Account samt Authtoken (https://dashboard.ngrok.com/get-started/your-authtoken).
Den Token unten eintragen oder als Colab-Secret (`NGROK_AUTH_TOKEN`) hinterlegen.

In [ ]:
from pyngrok import ngrok, conf

# Authtoken eintragen ODER als Colab-Secret "NGROK_AUTH_TOKEN" hinterlegen (empfohlen).
try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    NGROK_AUTH_TOKEN = ""  # <-- alternativ hier direkt als String eintragen

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

SERVER_PORT = 8081

# Bereits offene Tunnel schließen (falls Zelle erneut ausgeführt wird)
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(SERVER_PORT, "tcp")
public_url = tunnel.public_url  # z. B. "tcp://0.tcp.eu.ngrok.io:12345"

# In "host:port" umwandeln, das ist das Format, das flwr.client.start_client erwartet
address_without_scheme = public_url.replace("tcp://", "")

print("=" * 60)
print("Diese Adresse an die Studierenden weitergeben (SERVER_ADDRESS):")
print(f"  {address_without_scheme}")
print("=" * 60)

## 6. Logging: Metriken pro Runde

Damit nach jeder Runde `Runde`, `Global Loss` und `Global Accuracy` ausgegeben werden, definieren
wir eine kleine `FedAvg`-Unterklasse, die nach der Aggregation der Evaluationsergebnisse einer
Runde loggt und die Werte für die spätere Visualisierung sammelt. Außerdem werden
Aggregationsfunktionen für die von den Clients zurückgemeldeten Metriken (z. B. Accuracy)
definiert, da Flower diese nicht automatisch mittelt.

In [ ]:
from flwr.server.strategy import FedAvg
from flwr.common import Scalar
from typing import Optional


def weighted_average(metrics: list[tuple[int, dict[str, Scalar]]]) -> dict[str, Scalar]:
    """Aggregiert Client-Metriken gewichtet nach Anzahl lokaler Beispiele.

    Wird von Flower nach jeder Fit- bzw. Evaluate-Phase aufgerufen, um die von
    den einzelnen Clients gemeldeten Metriken (z. B. ``accuracy``) zu einer
    einzigen globalen Kennzahl zusammenzuführen.

    Args:
        metrics: Liste von Tupeln ``(num_examples, metrics_dict)``, eines pro
            Client, der an dieser Phase teilgenommen hat.

    Returns:
        dict[str, Scalar]: Ein Dictionary mit dem gewichteten Mittel jeder
        Metrik, die in den Client-Ergebnissen enthalten ist (z. B.
        ``{"accuracy": 0.93}``).
    """
    total_examples = sum(num_examples for num_examples, _ in metrics)
    if total_examples == 0:
        return {}

    aggregated: dict[str, Scalar] = {}
    metric_keys = metrics[0][1].keys()
    for key in metric_keys:
        aggregated[key] = (
            sum(num_examples * m[key] for num_examples, m in metrics) / total_examples
        )
    return aggregated


def fit_config(server_round: int) -> dict[str, Scalar]:
    """Erzeugt die Trainingskonfiguration, die pro Runde an jeden Client gesendet wird.

    Args:
        server_round: Die aktuelle Kommunikationsrunde (1-basiert).

    Returns:
        dict[str, Scalar]: Konfiguration mit der Anzahl lokaler Trainings-Epochen.
    """
    return {"local_epochs": CONFIG.local_epochs, "server_round": server_round}


class LoggingFedAvg(FedAvg):
    """FedAvg-Strategie, die nach jeder Runde Loss und Accuracy protokolliert.

    Erweitert :class:`flwr.server.strategy.FedAvg` um Konsolen-Logging und das
    Sammeln der globalen Accuracy pro Runde in :attr:`round_history`, sodass
    das Trainingsverlauf am Ende ohne erneute Auswertung geplottet werden
    kann.

    Attributes:
        round_history: Liste von ``(runde, loss, accuracy)``-Tupeln, eine
            pro abgeschlossener Kommunikationsrunde.
    """

    def __init__(self, *args, **kwargs) -> None:
        """Initialisiert die Strategie und das interne Verlaufs-Log.

        Args:
            *args: Positionsargumente, die an :class:`FedAvg` weitergereicht werden.
            **kwargs: Schlüsselwortargumente, die an :class:`FedAvg` weitergereicht werden.
        """
        super().__init__(*args, **kwargs)
        self.round_history: list[tuple[int, float, Optional[float]]] = []
        self.latest_parameters: Optional[fl.common.Parameters] = None

    def aggregate_fit(self, server_round, results, failures):
        """Aggregiert die Client-Gewichte einer Runde und merkt sie sich.

        Args:
            server_round: Die aktuelle Kommunikationsrunde (1-basiert).
            results: Liste erfolgreicher ``(ClientProxy, FitRes)``-Paare.
            failures: Liste fehlgeschlagener Trainingsanfragen dieser Runde.

        Returns:
            tuple: Wie :meth:`FedAvg.aggregate_fit` — aggregierte Parameter
            und aggregierte Metriken.
        """
        aggregated_parameters, aggregated_metrics = super().aggregate_fit(
            server_round, results, failures
        )
        if aggregated_parameters is not None:
            self.latest_parameters = aggregated_parameters
        return aggregated_parameters, aggregated_metrics

    def aggregate_evaluate(self, server_round, results, failures):
        """Aggregiert Evaluationsergebnisse und loggt Runde, Loss und Accuracy.

        Args:
            server_round: Die aktuelle Kommunikationsrunde (1-basiert).
            results: Liste erfolgreicher ``(ClientProxy, EvaluateRes)``-Paare.
            failures: Liste fehlgeschlagener Evaluationen dieser Runde.

        Returns:
            tuple: Wie :meth:`FedAvg.aggregate_evaluate` — aggregierter Loss
            und aggregierte Metriken.
        """
        aggregated_loss, aggregated_metrics = super().aggregate_evaluate(
            server_round, results, failures
        )
        accuracy = aggregated_metrics.get("accuracy") if aggregated_metrics else None
        self.round_history.append((server_round, aggregated_loss, accuracy))
        acc_str = f"{accuracy:.4f}" if accuracy is not None else "n/a"
        print(
            f"[Runde {server_round:2d}] Global Loss = {aggregated_loss:.4f} | "
            f"Global Accuracy = {acc_str}"
        )
        return aggregated_loss, aggregated_metrics


## 7. FedAvg-Strategie instanziieren

`min_fit_clients` und `min_available_clients` sollten für die echte Veranstaltung auf einen
sinnvollen Wert (z. B. die Hälfte der erwarteten Studierendenzahl) gesetzt werden, damit der
Server nicht auf einzelne, noch nicht verbundene Clients wartet.

In [ ]:
strategy = LoggingFedAvg(
    fraction_fit=CONFIG.fraction_fit,
    fraction_evaluate=CONFIG.fraction_evaluate,
    min_fit_clients=CONFIG.min_fit_clients,
    min_evaluate_clients=CONFIG.min_fit_clients,
    min_available_clients=CONFIG.min_available_clients,
    initial_parameters=initial_parameters,
    on_fit_config_fn=fit_config,
    fit_metrics_aggregation_fn=weighted_average,
    evaluate_metrics_aggregation_fn=weighted_average,
)


## 8. Server starten

Diese Zelle **blockiert**, bis entweder alle `num_rounds` Runden abgeschlossen sind oder ein
Fehler auftritt. Der Server lauscht lokal auf Port `8080`; über den in Schritt 5 geöffneten
ngrok-Tunnel ist dieser Port öffentlich unter der ausgegebenen Adresse erreichbar.

> Die Meldung `Using start_server() is deprecated` ist erwartet (siehe Hinweis oben) und kann
> ignoriert werden.

In [ ]:
history = fl.server.start_server(
    server_address=f"0.0.0.0:{SERVER_PORT}",
    config=fl.server.ServerConfig(num_rounds=CONFIG.num_rounds),
    strategy=strategy,
)


## 9. Visualisierung: Globale Accuracy über die Kommunikationsrunden

In [ ]:
import matplotlib.pyplot as plt
import os

rounds = [r for r, _, acc in strategy.round_history if acc is not None]
accuracies = [acc for _, _, acc in strategy.round_history if acc is not None]

fig = common.plot_accuracy_over_rounds(rounds, accuracies)

# Update the title to clarify it's validation accuracy
fig.axes[0].set_title("Globale Validierungs-Accuracy über Kommunikationsrunden")

# Create the 'figures' directory if it doesn't exist
if not os.path.exists("figures"):
    os.makedirs("figures")

fig.savefig("figures/global_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Finales Modell speichern

`LoggingFedAvg` hat sich in `aggregate_fit` die zuletzt aggregierten Gewichte gemerkt
(`strategy.latest_parameters`). Diese werden nun in das Keras-Modell geladen und gespeichert.

In [ ]:
from flwr.common import parameters_to_ndarrays

final_model = common.create_model()

if strategy.latest_parameters is not None:
    final_model.set_weights(parameters_to_ndarrays(strategy.latest_parameters))
    final_model.save("global_model.keras")
    print("Finales globales Modell gespeichert unter global_model.keras")
else:
    print("Keine aggregierten Parameter vorhanden -- wurde mindestens eine Runde abgeschlossen?")


## Erweiterungen (optional)

- **Nur ein Teil der Clients trainiert pro Runde:** `fraction_fit` in `CONFIG` auf z. B. `0.5`
  setzen.
- **Client-Ausfälle:** Der Server läuft weiter, solange `min_available_clients` erfüllt bleibt —
  Studierende können ihr Notebook einfach schließen, ohne den Betrieb zu stören.
- **Unterschiedliche lokale Trainingsdauer:** In `fit_config()` z. B. abhängig von
  `server_round` oder einer Client-Property unterschiedliche `local_epochs` zurückgeben.
- **Non-IID-Daten:** `CONFIG.partition_strategy` auf `"shard"` oder `"dominant"` setzen — muss mit
  der Einstellung in allen Client-Notebooks übereinstimmen!
- **Zentrales Training zum Vergleich:** `common.create_model()` einmal auf dem vollständigen
  MNIST-Trainingsdatensatz (ohne Partitionierung) trainieren und die erreichte Accuracy der
  federated erreichten Accuracy gegenüberstellen.